In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from scipy.special import expit as sigmoid

# 路径配置（Notebook 中建议手动指定 BASE_DIR，避免路径错误）
# 替换为你的实际工作目录（例如：BASE_DIR = Path("/Users/xxx/your_project")）
BASE_DIR = Path.cwd()  # 默认使用当前 Notebook 所在目录
OUT_DIR = BASE_DIR / "2_Data" / "Generate_Data"
FIG_DIR = BASE_DIR / "3_Figures" / "Generate_Data_v2.4_checks"

# 创建目录（不存在则创建）
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 验证路径（Notebook 中可选，用于确认路径正确）
print(f"数据保存目录: {OUT_DIR}")
print(f"图表保存目录: {FIG_DIR}")

In [2]:
# Hybrid GP + Sigmoid parameter generator (same as v2.1)
class HybridDDMParameterGenerator:
    def __init__(self, w=0.5):
        self.w = w
        kernel = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=1e-5)
        self.gp_v = GaussianProcessRegressor(kernel=kernel, normalize_y=True)
        self.gp_a = GaussianProcessRegressor(kernel=kernel, normalize_y=True)
        self.beta_v = np.array([0.01, 0.02, -0.01])
        self.beta_a = np.array([0.005, -0.01, 0.015])
    def sigmoid_part(self, X, beta):
        return sigmoid(X @ beta)
    def fit_gp(self, X, Y_v, Y_a):
        self.gp_v.fit(X, Y_v)
        self.gp_a.fit(X, Y_a)
    def predict_params(self, X):
        sig_v = self.sigmoid_part(X, self.beta_v)
        sig_a = self.sigmoid_part(X, self.beta_a)
        gp_v = self.gp_v.predict(X)
        gp_a = self.gp_a.predict(X)
        v = self.w * sig_v + (1 - self.w) * gp_v
        a = self.w * sig_a + (1 - self.w) * gp_a
        t0 = np.full(len(v), 0.2)
        z = 2 / a
        return v, a, t0, z

In [3]:
# Euler-style DDM simulator used by v2.1 & v2.3
def simulate_ddm_euler(v, a, z, t0, dt=0.001):
    """
    Return RT (s) and response (1=upper, 0=lower).
    """
    x = z
    t = 0.0
    while 0 < x < a:
        dx = v * dt + np.sqrt(dt) * np.random.randn()
        x += dx
        t += dt
    RT = t + t0
    response = 1 if x >= a else 0
    return RT, response

In [4]:
# S2-style analytic helpers (copied/adapted from S2 gen)
def k_P(P, k_min = 0.01, k_max = 0.15, gamma = 0.1, P0 = 32):
    return k_min + (k_max - k_min) / (1 + np.exp(-gamma * (P - P0)))
def v_P_Function(P, P1 = 4, k_min = 0.01, k_max = 0.15, gamma = 0.1, P0 = 32):
    k = k_P(P, k_min, k_max, gamma, P0)
    return 1 / (1 + np.exp(-k * (P - P1)))
def compute_v_s2(T, P, condition_key, alaph1=1.5, alaph2=-0.4, gamma=0.2):
    T_0 = 100
    k_T = 0.01
    v_T = 1 / (1 + np.exp(-k_T * (T - T_0)))
    v_P = v_P_Function(P = P, P1 = 4, k_min = 0.1, k_max = 0.05, gamma = gamma, P0 = 32)
    v_0 = v_T * v_P * 3
    if condition_key == 1:
        v_1 = v_0 * (1 + alaph1)
    else:
        v_1 = v_0 * (1 + alaph2)
    return v_1
def compute_a_s2(M, beta1=0.2, beta2=0, k=0.01, M_0=600):
    a_0 = 1 / (1 + np.exp(-k * (M - M_0))) * 3
    if M > 600:
        a_1 = a_0 * (1 + beta1)
    else:
        a_1 = a_0 * (1 + beta2)
    return a_1
def normalize_PTW_to_unit(P, T, W):
    # Map S2 ranges to [-1,1]
    P_norm = (P - 75.0) / 75.0
    T_norm = (T - 305.0) / 295.0
    W_norm = (W - 850.0) / 650.0
    return P_norm, T_norm, W_norm

In [5]:
# generate_dataset_s2: subject-level S2 sampling + GP correction
def generate_dataset_s2(n_subjects=5, trials_per_sub=50, w_gp=0.5, save_path="hybrid_ddm_output.csv", seed=None):
    """
    Generate data mixing S2 analytic rules and GP predictions.
    - Subject-level P/T/W sampled in original S2 ranges.
    - GP is trained in normalized [-1,1] space and mixed with S2 baseline.
    """
    if seed is not None:
        np.random.seed(seed)
    gen = HybridDDMParameterGenerator(w=0.5)
    # train GP on synthetic anchors (normalized space)
    X_train = np.random.uniform(-1, 1, size=(50, 3))
    Y_v = np.sin(X_train[:, 0]) * 0.5 + 0.1 * np.random.randn(50)
    Y_a = 1.5 + 0.3 * np.cos(X_train[:, 1]) + 0.05 * np.random.randn(50)
    gen.fit_gp(X_train, Y_v, Y_a)
    rows = []
    for subj in range(1, n_subjects + 1):
        T = np.random.randint(10, 600)
        P = np.random.randint(0, 150)
        W = np.random.randint(200, 1500)
        M = T + W
        a_s2 = compute_a_s2(M)
        while a_s2 <= 0:
            T = np.random.randint(10, 600)
            W = np.random.randint(200, 1500)
            M = T + W
            a_s2 = compute_a_s2(M)
        trials_per_condition = trials_per_sub // 2
        if trials_per_sub % 2 != 0:
            trials_per_condition += 1
        for condition_key in range(2):
            condition = "self" if condition_key == 1 else "stranger"
            trial_count = 0
            while trial_count < trials_per_condition:
                v_s2 = compute_v_s2(T, P, condition_key)
                Pn, Tn, Wn = normalize_PTW_to_unit(P, T, W)
                X = np.array([[Pn, Tn, Wn]])
                v_gp, a_gp, t0_arr, z_arr = gen.predict_params(X)
                v_mix = w_gp * v_gp[0] + (1 - w_gp) * v_s2
                a_mix = w_gp * a_gp[0] + (1 - w_gp) * a_s2
                v_final = np.random.normal(v_mix, 1.0)
                a_final = np.random.normal(a_mix, 0.5)
                if a_final <= 0:
                    a_final = max(0.1, a_mix)
                z_final = a_final / 2.0
                RT, response_bin = simulate_ddm_euler(v_final, a_final, z_final, 0.2)
                response = 1 if response_bin == 1 else 2
                rows.append([
                    v_final, a_final, 0.2, z_final,
                ])
                trial_count += 1
    df = pd.DataFrame(rows, columns=[
        "subject", "trial", "P", "T", "W", "M", "label",
        "v", "a", "t0", "z",
        "RT", "response"

_IncompleteInputError: incomplete input (3536430686.py, line 55)